In [ ]:
# Run this cell first to confirm all packages are available
import sys
print(f"Python : {sys.executable}")
print(f"Version: {sys.version.split()[0]}\n")

required = {"numpy": "numpy", "pandas": "pandas", "sklearn": "scikit-learn",
            "plotly": "plotly", "scipy": "scipy", "nbformat": "nbformat"}

for mod, pkg in required.items():
    try:
        m = __import__(mod)
        print(f"  OK  {pkg:<20} {getattr(m, '__version__', '?')}")
    except ImportError:
        print(f"  MISSING  {pkg}  <- pip install {pkg}")

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
    import xgboost
    print(f"  OK  xgboost              {xgboost.__version__}")
except Exception as e:
    XGBOOST_AVAILABLE = False
    print(f"  SKIPPED  xgboost  ({e})")

# Loan Approval — Preprocessing & Model Comparison

**Goal:** Clean and prepare the dataset, then train and compare seven classifiers.

**Sections:**
1. Imports & load data
2. Train / test split
3. Preprocessing — step by step
4. Random Forest baseline
5. Model comparison
6. Results — charts & confusion matrices
7. Feature importance

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path

# preprocessing & modelling
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

# metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, roc_curve, confusion_matrix,
)

# visualisation
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from IPython.display import display, HTML

RANDOM_STATE = 42
TEST_SIZE    = 0.20

## 1. Load Data

In [ ]:
data_path = Path("../dataset/loan_approval_dataset.csv")
df        = pd.read_csv(data_path, skipinitialspace=True)

print(f"Shape  : {df.shape}")
print(f"Columns: {df.columns.tolist()}")
display(df.head(3))

In [ ]:
# Encode target: Approved → 1, Rejected → 0
df["loan_status"] = (df["loan_status"].str.strip() == "Approved").astype(int)

print("Target value counts:")
print(df["loan_status"].value_counts())

## 2. Train / Test Split

Split the data **before** any preprocessing so that the test set is never seen during fitting.
`stratify=y` keeps the same class ratio (62% / 38%) in both splits.

In [ ]:
X = df.drop(columns=["loan_id", "loan_status"])
y = df["loan_status"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size    = TEST_SIZE,
    random_state = RANDOM_STATE,
    stratify     = y,
)

print(f"Train : {X_train.shape}  |  Test : {X_test.shape}")
print(f"Train target split: {y_train.value_counts(normalize=True).round(3).to_dict()}")
print(f"Test  target split: {y_test.value_counts(normalize=True).round(3).to_dict()}")

## 3. Preprocessing

Five steps applied in order. Each step that *learns* from the data (outlier bounds, scaler)
is **fit only on the training set** and then applied to both train and test.

| Step | What it does | Learns from train? |
|---|---|---|
| 1. Encode categoricals | `education`, `self_employed` → 0 / 1 | No |
| 2. Feature engineering | Add `total_assets`, `loan_to_income`, `asset_to_loan` | No |
| 3. Cap outliers | IQR method on asset columns | **Yes** |
| 4. Log-transform | `log1p` on right-skewed asset columns | No |
| 5. Scale features | `StandardScaler` — zero mean, unit variance | **Yes** |

### Step 1 — Encode Categorical Columns

In [ ]:
def encode_categoricals(df):
    df = df.copy()
    df["education"]     = (df["education"].str.strip()     == "Graduate").astype(int)
    df["self_employed"] = (df["self_employed"].str.strip() == "Yes").astype(int)
    return df

X_train = encode_categoricals(X_train)
X_test  = encode_categoricals(X_test)

print("education unique values    :", X_train["education"].unique())
print("self_employed unique values:", X_train["self_employed"].unique())

### Step 2 — Feature Engineering

The financial columns are highly correlated (income ↔ loan_amount at 0.93).
Instead of dropping them, we create three composite features that capture
meaningful ratios, giving the model richer signal.

In [ ]:
def add_features(df):
    df = df.copy()
    df["total_assets"]   = (df["residential_assets_value"]
                           + df["commercial_assets_value"]
                           + df["luxury_assets_value"]
                           + df["bank_asset_value"])
    df["loan_to_income"] = df["loan_amount"] / df["income_annum"]
    df["asset_to_loan"]  = df["total_assets"] / df["loan_amount"]
    return df

X_train = add_features(X_train)
X_test  = add_features(X_test)

print(f"Columns after feature engineering ({X_train.shape[1]}):")
print(X_train.columns.tolist())

### Step 3 — Cap Outliers (IQR Method)

Learn the IQR bounds **from training data only**, then apply them to both splits.
This prevents any information from the test set leaking into the preprocessing.

In [ ]:
CAP_COLS = ["residential_assets_value", "commercial_assets_value", "bank_asset_value"]

# Learn bounds from training data
cap_bounds = {}
for col in CAP_COLS:
    q1  = X_train[col].quantile(0.25)
    q3  = X_train[col].quantile(0.75)
    iqr = q3 - q1
    cap_bounds[col] = (q1 - 1.5 * iqr, q3 + 1.5 * iqr)
    print(f"  {col}: lower={cap_bounds[col][0]:,.0f}  upper={cap_bounds[col][1]:,.0f}")

# Apply to both splits
for col, (lo, hi) in cap_bounds.items():
    X_train[col] = X_train[col].clip(lower=lo, upper=hi)
    X_test[col]  = X_test[col].clip(lower=lo, upper=hi)

print("\nOutlier capping applied.")

### Step 4 — Log-Transform Skewed Columns

`residential_assets_value` and `commercial_assets_value` are right-skewed (skew ~0.97).
`np.log1p` compresses large values and makes the distribution more symmetric.

`np.maximum(x, 0)` clips negatives to 0 first — some rows have negative asset values
(representing debt), and `log1p` is undefined for values below −1.

In [ ]:
SKEWED_COLS = ["residential_assets_value", "commercial_assets_value", "bank_asset_value"]

for col in SKEWED_COLS:
    X_train[col] = np.log1p(np.maximum(X_train[col], 0))
    X_test[col]  = np.log1p(np.maximum(X_test[col],  0))

print("Skewness after log transform:")
print(X_train[SKEWED_COLS].skew().round(3))

### Step 5 — Feature Scaling

`StandardScaler` transforms each feature to zero mean and unit variance.
- **Fit** on `X_train` only — learns the mean and std from training data
- **Transform** both `X_train` and `X_test` using those same values

Required for Logistic Regression, SVM, KNN. Harmless for tree-based models.

In [ ]:
scaler       = StandardScaler()
X_train_proc = scaler.fit_transform(X_train)   # fit + transform on train
X_test_proc  = scaler.transform(X_test)        # transform only on test

feature_names = X_train.columns.tolist()

print(f"Final shape — Train: {X_train_proc.shape}  |  Test: {X_test_proc.shape}")
print(f"\nFeatures ({len(feature_names)}): {feature_names}")
print(f"\nNaN check — Train: {np.isnan(X_train_proc).sum()}  |  Test: {np.isnan(X_test_proc).sum()}")

## 4. Evaluation Helper

A single function that trains a model and returns all metrics.
Reused for both the baseline and the full comparison.

In [ ]:
def evaluate_model(name, model, X_tr, X_te, y_tr, y_te):
    model.fit(X_tr, y_tr)

    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:, 1] if hasattr(model, "predict_proba") else None

    metrics = {
        "Model":     name,
        "Accuracy":  round(accuracy_score(y_te, y_pred),  4),
        "Precision": round(precision_score(y_te, y_pred), 4),
        "Recall":    round(recall_score(y_te, y_pred),    4),
        "F1":        round(f1_score(y_te, y_pred),        4),
        "ROC-AUC":   round(roc_auc_score(y_te, y_prob),   4) if y_prob is not None else None,
    }
    return metrics, model, y_pred, y_prob

print("evaluate_model() ready.")

## 5. Baseline — Random Forest

Random Forest is the starting point because:
- Works well out of the box without scaling or tuning
- Handles correlated features and mixed scales gracefully
- Gives feature importance for free

In [ ]:
rf_metrics, rf_model, rf_pred, rf_prob = evaluate_model(
    "Random Forest",
    RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE),
    X_train_proc, X_test_proc, y_train, y_test,
)

print("=" * 45)
print("  RANDOM FOREST — BASELINE RESULTS")
print("=" * 45)
for k, v in rf_metrics.items():
    if k != "Model":
        print(f"  {k:<12} {v:.4f}")
print("=" * 45)

## 6. Model Comparison

All seven models use **default hyperparameters** — the goal is a fair baseline
comparison before any tuning.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Decision Tree":       DecisionTreeClassifier(random_state=RANDOM_STATE),
    "Random Forest":       RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE),
    "Gradient Boosting":   GradientBoostingClassifier(n_estimators=100, random_state=RANDOM_STATE),
    "SVM":                 SVC(probability=True, random_state=RANDOM_STATE),
    "KNN":                 KNeighborsClassifier(n_neighbors=5),
}

# add XGBoost only if available (requires 64-bit Python)
if XGBOOST_AVAILABLE:
    from xgboost import XGBClassifier
    models["XGBoost"] = XGBClassifier(n_estimators=100, random_state=RANDOM_STATE,
                                       eval_metric="logloss", verbosity=0)

results      = {}   # stores (metrics, model, y_pred, y_prob) per model name
metrics_rows = []

print("Training models...\n")
for name, model in models.items():
    m, fitted_model, y_pred, y_prob = evaluate_model(
        name, model, X_train_proc, X_test_proc, y_train, y_test
    )
    results[name]   = (m, fitted_model, y_pred, y_prob)
    metrics_rows.append(m)
    print(f"  {name:<25}  ROC-AUC={m['ROC-AUC']:.4f}   F1={m['F1']:.4f}")

print("\nDone.")

### Results Table

In [ ]:
results_df = (pd.DataFrame(metrics_rows)
              .set_index("Model")
              .sort_values("ROC-AUC", ascending=False))

print(results_df.to_string())

### Comparison Chart

In [ ]:
COLORS = px.colors.qualitative.Plotly

fig = go.Figure()
for i, metric in enumerate(["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]):
    fig.add_trace(go.Bar(
        name         = metric,
        x            = results_df.index.tolist(),
        y            = results_df[metric].values,
        marker_color = COLORS[i],
        text         = [f"{v:.3f}" for v in results_df[metric].values],
        textposition = "outside",
        textfont     = dict(size=10),
    ))

fig.update_layout(
    title      = dict(text="Model Comparison — All Metrics",
                      font=dict(size=18, color="#1E3A5F"), x=0.5, xanchor="center"),
    barmode    = "group",
    template   = "plotly_white",
    height     = 520,
    yaxis      = dict(range=[0, 1.12], showgrid=False, title="Score"),
    xaxis      = dict(showgrid=False),
    legend     = dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    font       = dict(family="Arial, sans-serif", size=12),
    margin     = dict(t=100, b=60, l=60, r=40),
)
fig.show()

### ROC Curves

In [ ]:
fig = go.Figure()

for i, (name, (m, _, _, y_prob)) in enumerate(results.items()):
    if y_prob is not None:
        fpr, tpr, _ = roc_curve(y_test, y_prob)
        fig.add_trace(go.Scatter(
            x    = fpr, y = tpr, mode = "lines",
            name = f"{name}  (AUC={m['ROC-AUC']:.3f})",
            line = dict(color=COLORS[i % len(COLORS)], width=2.5),
        ))

# random classifier baseline
fig.add_trace(go.Scatter(
    x=[ 0, 1], y=[0, 1], mode="lines",
    name="Random classifier",
    line=dict(color="#9CA3AF", width=1.5, dash="dash"),
))

fig.update_layout(
    title    = dict(text="ROC Curves — All Models",
                    font=dict(size=18, color="#1E3A5F"), x=0.5, xanchor="center"),
    template = "plotly_white",
    height   = 520,
    xaxis    = dict(title="False Positive Rate", showgrid=False),
    yaxis    = dict(title="True Positive Rate",  showgrid=False, range=[0, 1.02]),
    legend   = dict(x=0.62, y=0.08, bgcolor="white", bordercolor="#E5E7EB", borderwidth=1),
    font     = dict(family="Arial, sans-serif", size=12),
    margin   = dict(t=80, b=60, l=70, r=40),
)
fig.show()

### Confusion Matrices

- **Top-left (TN):** Correctly predicted Rejected  
- **Top-right (FP):** Wrongly approved (should have been rejected)  
- **Bottom-left (FN):** Wrongly rejected (should have been approved)  
- **Bottom-right (TP):** Correctly predicted Approved

In [ ]:
model_names = list(results.keys())
ncols       = min(4, len(model_names))
nrows       = -(-len(model_names) // ncols)
labels      = ["Rejected (0)", "Approved (1)"]

fig = make_subplots(
    rows              = nrows,
    cols              = ncols,
    subplot_titles    = model_names,
    vertical_spacing  = 0.14,
    horizontal_spacing= 0.06,
)

for i, name in enumerate(model_names):
    _, _, y_pred, _ = results[name]
    cm  = confusion_matrix(y_test, y_pred)
    row = i // ncols + 1
    col = i %  ncols + 1

    fig.add_trace(
        go.Heatmap(
            z            = cm,
            x            = labels,
            y            = labels,
            colorscale   = "Blues",
            showscale    = False,
            text         = cm,
            texttemplate = "%{text}",
            textfont     = dict(size=14),
        ),
        row=row, col=col,
    )

fig.update_layout(
    title    = dict(text="Confusion Matrices — All Models",
                    font=dict(size=18, color="#1E3A5F"), x=0.5, xanchor="center"),
    template = "plotly_white",
    height   = 320 * nrows,
    font     = dict(family="Arial, sans-serif", size=11),
    margin   = dict(t=80, b=40, l=60, r=40),
)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False)
fig.show()

### Feature Importance — Random Forest

Shows which features the Random Forest relied on most when splitting.
Higher = more important for predicting loan approval.

In [ ]:
_, rf_fitted, _, _ = results["Random Forest"]

importance_df = pd.DataFrame({
    "Feature":    feature_names,
    "Importance": rf_fitted.feature_importances_,
}).sort_values("Importance", ascending=True)

fig = go.Figure(go.Bar(
    x            = importance_df["Importance"],
    y            = importance_df["Feature"],
    orientation  = "h",
    marker       = dict(color=importance_df["Importance"],
                        colorscale="Blues", showscale=False),
    text         = [f"{v:.4f}" for v in importance_df["Importance"]],
    textposition = "outside",
    textfont     = dict(size=11, color="#1F2937"),
))

fig.update_layout(
    title    = dict(text="Random Forest — Feature Importance",
                    font=dict(size=18, color="#1E3A5F"), x=0.5, xanchor="center"),
    template = "plotly_white",
    height   = max(400, len(feature_names) * 35),
    xaxis    = dict(title="Importance Score", showgrid=False),
    yaxis    = dict(showgrid=False),
    font     = dict(family="Arial, sans-serif", size=12),
    margin   = dict(t=80, b=60, l=200, r=100),
)
fig.show()

## 7. Summary & Next Steps

### Key Observations
- **CIBIL score** dominates feature importance — as EDA predicted
- Tree-based models (RF, GB, XGBoost) score near-perfect due to clean CIBIL score separation
- Logistic Regression and SVM show lower scores — worth tuning regularisation

### Next Steps
| Step | What to do |
|---|---|
| Hyperparameter tuning | `GridSearchCV` or `RandomizedSearchCV` on best 2–3 models |
| Threshold tuning | Adjust decision threshold from 0.5 using Precision-Recall curve |
| Cross-validation | Replace single split with `StratifiedKFold(n_splits=5)` |
| Feature selection | Drop near-zero importance features, re-evaluate |
| Production pipeline | Wrap all preprocessing + best model in a `sklearn.Pipeline`, save with `joblib` |

In [ ]:
best_name    = results_df["ROC-AUC"].idxmax()
best_metrics = results_df.loc[best_name]

print(f"Best model      : {best_name}")
print(f"  ROC-AUC       : {best_metrics['ROC-AUC']:.4f}")
print(f"  F1-Score      : {best_metrics['F1']:.4f}")
print(f"  Accuracy      : {best_metrics['Accuracy']:.4f}")
print(f"\nRandom Forest baseline:")
print(f"  ROC-AUC       : {results_df.loc['Random Forest', 'ROC-AUC']:.4f}")
print(f"  F1-Score      : {results_df.loc['Random Forest', 'F1']:.4f}")